# 005 GPM / ND Degradation Root-Cause Diagnostic Analysis

**Goal**: use data-driven geographic analysis to identify the true causes of degradation for GPM and ND in specific regions.

**Summary of degradation patterns**:

- **GPM stable degradation (4 regions)**: TLC1 (+3-6%), TLD4 (+2-5%), TLF1 (+0.5-1.5%), TLF2 (+0.5-1.2%)
- **ND stable degradation (4 regions)**: London (+2.5-7%), TLH1 (+2-6%), TLH2 (+3-11%), TLH3 (+3-8%)

**Data source**: uses only existing pickle/gpkg data; allocation is not re-run.

In [ ]:
# Cell 1: Load data
import sys
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from scipy import stats
from scipy.spatial.distance import cdist

PROJECT_ROOT = Path('../../').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from SpatialAllocation.utils.NetworkDistance import load_distance_results

warnings.filterwarnings('ignore', category=FutureWarning)

# ─── Path constants ───
DATA_DIR = Path('./results/intermediate')
ASSEMBLED_DIR = DATA_DIR / 'features' / 'assembled'
ND_DATA_DIR = DATA_DIR / 'features' / 'network_distance'
OUTPUT_DIR = Path('./results/degradation_analysis')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STUDY_REGIONS = [
    'London', 'TLC1', 'TLC2', 'TLD3', 'TLD4', 'TLD6',
    'TLE3', 'TLE4', 'TLF1', 'TLF2', 'TLG1', 'TLG2',
    'TLH1', 'TLH2', 'TLH3', 'TLJ1',
]

LU_COLS = [
    'lu_residential_prop', 'lu_commercial_prop',
    'lu_industrial_prop', 'lu_agricultural_prop', 'lu_others_prop',
]
LU_LABELS = ['residential', 'commercial', 'industrial', 'agricultural', 'others']
PCT_COLS = [
    'residential_percent', 'commercial_percent',
    'industrial_percent', 'agricultural_percent', 'others_percent',
]

# ─── Load global data ───
region_gdf = gpd.read_file(str(DATA_DIR / 'ITL3_region.gpkg'))
substations_gdf = gpd.read_file(str(DATA_DIR / 'substations.gpkg'))

# ─── Load RMSE results ───
rmse_eu = pd.read_csv('./results/static_allocation/all_regions_rmse.csv', index_col=0)
rmse_nd = pd.read_csv('./results/static_allocation_nd/all_regions_rmse.csv', index_col=0)

# ─── Compute GPM delta (categorical mode, average change across 4 bases) ───
gpm_delta = {}
for region in STUDY_REGIONS:
    deltas = []
    deltas.append(rmse_eu.loc['voronoi_gpm', region] - rmse_eu.loc['voronoi', region])
    deltas.append(rmse_eu.loc['civd_gpm', region] - rmse_eu.loc['civd', region])
    deltas.append(rmse_nd.loc['voronoi_gpm_ND', region] - rmse_nd.loc['voronoi_ND', region])
    deltas.append(rmse_nd.loc['civd_gpm_ND', region] - rmse_nd.loc['civd_ND', region])
    gpm_delta[region] = np.mean(deltas)

# ─── Compute ND delta (average change across 4 bases) ───
nd_delta = {}
for region in STUDY_REGIONS:
    deltas = []
    deltas.append(rmse_nd.loc['voronoi_ND', region] - rmse_eu.loc['voronoi', region])
    deltas.append(rmse_nd.loc['civd_ND', region] - rmse_eu.loc['civd', region])
    deltas.append(rmse_nd.loc['voronoi_gpm_ND', region] - rmse_eu.loc['voronoi_gpm', region])
    deltas.append(rmse_nd.loc['civd_gpm_ND', region] - rmse_eu.loc['civd_gpm', region])
    nd_delta[region] = np.mean(deltas)

# ─── Load grid_gdf for all regions ───
grids = {}
for loc in STUDY_REGIONS:
    path = ASSEMBLED_DIR / f'{loc}_grid_points.pickle'
    with open(path, 'rb') as f:
        grid_gdf, step_size_m = pickle.load(f)
    grids[loc] = (grid_gdf, step_size_m)

print(f'Loaded: {len(grids)} region grid_gdf')
print(f'GPM degraded regions: {[r for r, d in gpm_delta.items() if d > 0]}')
print(f'ND degraded regions: {[r for r, d in nd_delta.items() if d > 0]}')

## Cell 2: GPM Degradation Diagnosis — Land-Use Homogeneity vs. Electricity-Use-Structure Match

**Hypothesis**: the categorical mode degrades to uniform in regions with highly homogeneous land use, because winner-take-all causes all agents to receive the same score.

For each ITL3 within each region, compute:
1. **Dominant-type coverage (homogeneity)**: the proportion of agents sharing the same dominant type
2. **Land-use entropy (Shannon entropy)**: the information content of the region-level land-use distribution
3. **Number of unique categorical score values**: high homogeneity → only 1-2 distinct scores
4. **Match between dominant type and electricity-use structure**: the score magnitude reflects allocation effectiveness

In [ ]:
# Cell 2: GPM Degradation Diagnosis — Land-Use Homogeneity vs. Electricity-Use-Structure Match

gpm_diag_rows = []

for loc in STUDY_REGIONS:
    grid_gdf, _ = grids[loc]
    study_itl3 = grid_gdf['ITL3'].unique()
    region_sub = region_gdf[region_gdf['ITL3'].isin(study_itl3)].copy()
    region_info = region_sub.set_index('ITL3')

    # Dominant land-use type per agent
    lu_vals = grid_gdf[LU_COLS].values
    dominant_idx = np.argmax(lu_vals, axis=1)  # 0~4

    # ── Region-level metrics (weighted average across ITL3s) ──
    itl3_homogeneities = []
    itl3_entropies = []
    itl3_n_unique_scores = []
    itl3_dominant_types = []
    itl3_dominant_score_vals = []
    itl3_weights = []  # weighted by agent count

    for itl3 in study_itl3:
        mask = grid_gdf['ITL3'] == itl3
        n_agents_itl3 = mask.sum()
        if n_agents_itl3 == 0:
            continue

        dom_in_itl3 = dominant_idx[mask]

        # 1. Homogeneity: share of the most common dominant type
        mode_result = stats.mode(dom_in_itl3, keepdims=True)
        mode_type = mode_result.mode[0]
        homogeneity = (dom_in_itl3 == mode_type).mean()

        # 2. Shannon entropy (based on region-level land-use means)
        lu_means = grid_gdf.loc[mask, LU_COLS].mean().values
        lu_means = lu_means[lu_means > 0]  # drop zeros
        entropy = -np.sum(lu_means * np.log(lu_means + 1e-12))

        # 3. Number of unique categorical score values
        if itl3 in region_info.index:
            pcts = np.array([region_info.loc[itl3, c] for c in PCT_COLS])
            scores = pcts[dom_in_itl3]  # categorical score for each agent
            n_unique = len(np.unique(np.round(scores, 6)))
            dominant_type_name = LU_LABELS[mode_type]
            dominant_score = pcts[mode_type]
        else:
            n_unique = 0
            dominant_type_name = 'unknown'
            dominant_score = 0.0

        itl3_homogeneities.append(homogeneity)
        itl3_entropies.append(entropy)
        itl3_n_unique_scores.append(n_unique)
        itl3_dominant_types.append(dominant_type_name)
        itl3_dominant_score_vals.append(dominant_score)
        itl3_weights.append(n_agents_itl3)

    weights = np.array(itl3_weights, dtype=float)
    weights /= weights.sum()

    avg_homogeneity = np.average(itl3_homogeneities, weights=weights)
    avg_entropy = np.average(itl3_entropies, weights=weights)
    avg_n_unique = np.average(itl3_n_unique_scores, weights=weights)
    # Use the mode as the region's dominant type
    from collections import Counter
    type_counts = Counter(itl3_dominant_types)
    region_dominant_type = type_counts.most_common(1)[0][0]
    avg_dominant_score = np.average(itl3_dominant_score_vals, weights=weights)

    gpm_diag_rows.append({
        'region': loc,
        'homogeneity': avg_homogeneity,
        'entropy': avg_entropy,
        'n_unique_scores': avg_n_unique,
        'dominant_type': region_dominant_type,
        'dominant_score': avg_dominant_score,
        'gpm_delta_rmse': gpm_delta[loc],
        'is_degraded': gpm_delta[loc] > 0,
    })

gpm_diag_df = pd.DataFrame(gpm_diag_rows).set_index('region')

print('=== GPM Degradation Diagnostic Table ===')
display(gpm_diag_df.round(4))

# ── Correlation analysis ──
corr_homo, p_homo = stats.pearsonr(gpm_diag_df['homogeneity'], gpm_diag_df['gpm_delta_rmse'])
corr_entropy, p_entropy = stats.pearsonr(gpm_diag_df['entropy'], gpm_diag_df['gpm_delta_rmse'])
corr_nunique, p_nunique = stats.pearsonr(gpm_diag_df['n_unique_scores'], gpm_diag_df['gpm_delta_rmse'])

print(f'\nCorrelation (Pearson):')
print(f'  homogeneity vs GPM delta: r={corr_homo:.3f}, p={p_homo:.4f}')
print(f'  entropy vs GPM delta:     r={corr_entropy:.3f}, p={p_entropy:.4f}')
print(f'  n_unique vs GPM delta:    r={corr_nunique:.3f}, p={p_nunique:.4f}')

In [ ]:
# Cell 3: GPM Degradation Diagnosis — Electricity-Use-Structure Match Details

print('=== Per-region GPM categorical score analysis ===\n')

gpm_score_detail_rows = []

for loc in STUDY_REGIONS:
    grid_gdf, _ = grids[loc]
    study_itl3 = grid_gdf['ITL3'].unique()
    region_sub = region_gdf[region_gdf['ITL3'].isin(study_itl3)].copy()
    region_info = region_sub.set_index('ITL3')

    lu_vals = grid_gdf[LU_COLS].values
    dominant_idx = np.argmax(lu_vals, axis=1)

    # Count agents per dominant type
    type_counts = {}
    for i, label in enumerate(LU_LABELS):
        type_counts[label] = (dominant_idx == i).sum()

    total = len(grid_gdf)
    type_pcts = {k: v / total for k, v in type_counts.items()}

    # Weighted average score (across ITL3s)
    all_scores = []
    for itl3 in study_itl3:
        mask = grid_gdf['ITL3'] == itl3
        if mask.sum() == 0 or itl3 not in region_info.index:
            continue
        pcts = np.array([region_info.loc[itl3, c] for c in PCT_COLS])
        scores = pcts[dominant_idx[mask]]
        all_scores.extend(scores)

    all_scores = np.array(all_scores)
    score_cv = all_scores.std() / (all_scores.mean() + 1e-12)  # coefficient of variation

    gpm_score_detail_rows.append({
        'region': loc,
        'pct_residential': type_pcts['residential'],
        'pct_commercial': type_pcts['commercial'],
        'pct_industrial': type_pcts['industrial'],
        'pct_agricultural': type_pcts['agricultural'],
        'pct_others': type_pcts['others'],
        'score_mean': all_scores.mean(),
        'score_std': all_scores.std(),
        'score_cv': score_cv,
        'gpm_delta': gpm_delta[loc],
    })

gpm_score_df = pd.DataFrame(gpm_score_detail_rows).set_index('region')
display(gpm_score_df.round(4))

# Correlation between score_cv and GPM delta
corr_cv, p_cv = stats.pearsonr(gpm_score_df['score_cv'], gpm_score_df['gpm_delta'])
print(f'\nscore_cv vs GPM delta: r={corr_cv:.3f}, p={p_cv:.4f}')
print('Smaller score_cv → less score variation across agents → GPM closer to uniform → more likely to degrade')

## Cell 4: ND Degradation Diagnosis — Road-Network / Euclidean Distance Ratio

**Hypothesis**: ND degrades in road-network-dense urban regions because `ratio = nd / euclidean ≈ constant`, which does not change the agent-target ranking.

Key metrics:
- **ratio_median**: median road-network detour factor (closer to 1.0 → ND carries less new information)
- **ratio_cv**: coefficient of variation of the ratio (smaller → ND changes the ranking less)

In [ ]:
# Cell 4: ND Degradation Diagnosis — Road-Network / Euclidean Distance Ratio (vectorized)

def reconstruct_full_nd_matrix(nd_matrix, nd_target_indices, n_targets):
    """Sparse (N, K=20) -> dense (N, M), missing entries filled with inf."""
    n_agents, k = nd_matrix.shape
    full = np.full((n_agents, n_targets), np.inf, dtype=np.float64)
    rows = np.repeat(np.arange(n_agents), k)
    cols = nd_target_indices.ravel()
    vals = nd_matrix.ravel()
    valid = cols >= 0
    full[rows[valid], cols[valid]] = vals[valid]
    return full


nd_diag_rows = []
nd_ratio_distributions = {}

for loc in STUDY_REGIONS:
    print(f'  Processing {loc}...', end=' ')
    grid_gdf, _ = grids[loc]
    study_itl3 = grid_gdf['ITL3'].unique()
    subs_sub = substations_gdf[substations_gdf['ITL3'].isin(study_itl3)].copy().reset_index(drop=True)

    # Load road-network distances
    nd_dir = ND_DATA_DIR / loc
    nd_matrix, nd_target_idx, _, nd_target_map = load_distance_results(str(nd_dir))
    full_nd = reconstruct_full_nd_matrix(nd_matrix, nd_target_idx, len(subs_sub))

    # Euclidean distance (uniformly projected to EPSG:27700 to ensure units are meters)
    grid_proj = grid_gdf.to_crs('EPSG:27700')
    subs_proj = subs_sub.to_crs('EPSG:27700')
    grid_coords = np.column_stack([grid_proj.geometry.x.values, grid_proj.geometry.y.values])
    target_coords = np.column_stack([subs_proj.geometry.x.values, subs_proj.geometry.y.values])
    euclidean_full = cdist(grid_coords, target_coords, metric='euclidean')  # units: meters

    # Vectorized ratio computation: take the Euclidean distances corresponding to the K=20 nearest neighbors
    n_agents, k = nd_matrix.shape
    row_idx = np.repeat(np.arange(n_agents), k)
    col_idx = nd_target_idx.ravel()
    nd_vals = nd_matrix.ravel()

    valid = (col_idx >= 0) & np.isfinite(nd_vals)
    eu_vals = euclidean_full[row_idx[valid], col_idx[valid]]
    eu_nonzero = eu_vals > 0
    ratios_arr = nd_vals[valid][eu_nonzero] / eu_vals[eu_nonzero]
    nd_ratio_distributions[loc] = ratios_arr

    ratio_median = np.median(ratios_arr)
    ratio_mean = np.mean(ratios_arr)
    ratio_std = np.std(ratios_arr)
    ratio_cv = ratio_std / (ratio_mean + 1e-12)
    pct_near_straight = (ratios_arr < 1.2).mean()

    nd_diag_rows.append({
        'region': loc,
        'ratio_median': ratio_median,
        'ratio_mean': ratio_mean,
        'ratio_std': ratio_std,
        'ratio_cv': ratio_cv,
        'pct_near_straight': pct_near_straight,
        'n_targets': len(subs_sub),
        'nd_delta_rmse': nd_delta[loc],
        'is_degraded': nd_delta[loc] > 0,
    })
    print(f'ratio_median={ratio_median:.3f}, ratio_cv={ratio_cv:.3f}')

nd_diag_df = pd.DataFrame(nd_diag_rows).set_index('region')

print('\n=== ND Degradation Diagnostic Table ===')
display(nd_diag_df.round(4))

# Correlation analysis
corr_ratio_cv, p_ratio_cv = stats.pearsonr(nd_diag_df['ratio_cv'], nd_diag_df['nd_delta_rmse'])
corr_ratio_med, p_ratio_med = stats.pearsonr(nd_diag_df['ratio_median'], nd_diag_df['nd_delta_rmse'])
corr_near, p_near = stats.pearsonr(nd_diag_df['pct_near_straight'], nd_diag_df['nd_delta_rmse'])

print(f'\nCorrelation (Pearson):')
print(f'  ratio_cv vs ND delta:          r={corr_ratio_cv:.3f}, p={p_ratio_cv:.4f}')
print(f'  ratio_median vs ND delta:      r={corr_ratio_med:.3f}, p={p_ratio_med:.4f}')
print(f'  pct_near_straight vs ND delta: r={corr_near:.3f}, p={p_near:.4f}')

## Cell 5: ND Degradation Diagnosis — Assignment Change Analysis

Compares the difference between Euclidean Voronoi assignment and road-network Voronoi assignment:
- **change_rate**: the proportion of agents whose assignment changed
- High change_rate + degradation → road-network distance changed the ranking, but in the wrong direction

In [ ]:
# Cell 5: ND Degradation Diagnosis — Assignment Change Analysis

assign_rows = []

for loc in STUDY_REGIONS:
    grid_gdf, _ = grids[loc]
    study_itl3 = grid_gdf['ITL3'].unique()
    subs_sub = substations_gdf[substations_gdf['ITL3'].isin(study_itl3)].copy().reset_index(drop=True)

    # Load road-network distances
    nd_dir = ND_DATA_DIR / loc
    nd_matrix, nd_target_idx, _, _ = load_distance_results(str(nd_dir))
    full_nd = reconstruct_full_nd_matrix(nd_matrix, nd_target_idx, len(subs_sub))

    # Euclidean distance (uniformly projected to EPSG:27700 to ensure units are meters)
    grid_proj = grid_gdf.to_crs('EPSG:27700')
    subs_proj = subs_sub.to_crs('EPSG:27700')
    grid_coords = np.column_stack([grid_proj.geometry.x.values, grid_proj.geometry.y.values])
    target_coords = np.column_stack([subs_proj.geometry.x.values, subs_proj.geometry.y.values])
    euclidean_full = cdist(grid_coords, target_coords, metric='euclidean')  # units: meters

    # Assignment
    eu_assignment = euclidean_full.argmin(axis=1)
    nd_assignment = full_nd.argmin(axis=1)

    # Change rate
    changed = eu_assignment != nd_assignment
    change_rate = changed.mean()

    # For changed agents, analyze the direction of change
    actual_demands = subs_sub['Demand (MVA)'].values
    if changed.sum() > 0:
        old_target_demands = actual_demands[eu_assignment[changed]]
        new_target_demands = actual_demands[nd_assignment[changed]]
        demand_change_mean = (new_target_demands - old_target_demands).mean()
    else:
        demand_change_mean = 0.0

    assign_rows.append({
        'region': loc,
        'change_rate': change_rate,
        'n_changed': changed.sum(),
        'n_total': len(grid_gdf),
        'demand_change_mean': demand_change_mean,
        'nd_delta_rmse': nd_delta[loc],
        'is_degraded': nd_delta[loc] > 0,
    })

assign_df = pd.DataFrame(assign_rows).set_index('region')

print('=== ND Assignment Change Analysis ===')
display(assign_df.round(4))

corr_change, p_change = stats.pearsonr(assign_df['change_rate'], assign_df['nd_delta_rmse'])
print(f'\nchange_rate vs ND delta: r={corr_change:.3f}, p={p_change:.4f}')

In [ ]:
# Cell 6: Overall Summary + Data Export

# ── Best explanatory factor for GPM ──
gpm_factors = {
    'homogeneity': (corr_homo, p_homo),
    'entropy': (corr_entropy, p_entropy),
    'score_cv': (corr_cv, p_cv),
}
best_gpm = max(gpm_factors, key=lambda k: abs(gpm_factors[k][0]))

# ── Best explanatory factor for ND ──
nd_factors = {
    'ratio_cv': (corr_ratio_cv, p_ratio_cv),
    'ratio_median': (corr_ratio_med, p_ratio_med),
    'pct_near_straight': (corr_near, p_near),
    'change_rate': (corr_change, p_change),
}
best_nd = max(nd_factors, key=lambda k: abs(nd_factors[k][0]))

print('=' * 60)
print('  GPM / ND Degradation Diagnosis — Overall Conclusions')
print('=' * 60)

print(f'\n--- GPM ---')
print(f'Best explanatory factor: {best_gpm} (r={gpm_factors[best_gpm][0]:.3f}, p={gpm_factors[best_gpm][1]:.4f})')
for name, (r, p) in gpm_factors.items():
    print(f'  {name:20s}: r={r:.3f}, p={p:.4f}')

print(f'\n--- ND ---')
print(f'Best explanatory factor: {best_nd} (r={nd_factors[best_nd][0]:.3f}, p={nd_factors[best_nd][1]:.4f})')
for name, (r, p) in nd_factors.items():
    print(f'  {name:20s}: r={r:.3f}, p={p:.4f}')

# ── Save diagnostic data ──
gpm_diag_df.to_csv(OUTPUT_DIR / 'gpm_diagnostic_table.csv')
gpm_score_df.to_csv(OUTPUT_DIR / 'gpm_score_detail_table.csv')
nd_diag_df.to_csv(OUTPUT_DIR / 'nd_diagnostic_table.csv')
assign_df.to_csv(OUTPUT_DIR / 'nd_assignment_change_table.csv')

print(f'\nSaved all diagnostic data to: {OUTPUT_DIR}')
for f in sorted(OUTPUT_DIR.glob('*.csv')):
    print(f'  {f.name}')

## 3. Correlation Pattern Analysis

**Goal**: analyze the Pearson correlation performance of each method, and the impact of GPM/ND on correlation.

Correlation measures "how consistent the predicted substation load ranking is with the actual ranking," complementing RMSE (absolute error):
- **High corr + high RMSE**: ranking is correct but the offset is large
- **Low corr + low RMSE**: all stations receive similar values (mean-like strategy)
- **High corr + low RMSE**: ideal state

In [ ]:
# Cell 7: Load corr data + average corr ranking per method

corr_eu = pd.read_csv('./results/static_allocation/all_regions_corr.csv', index_col=0)
corr_nd = pd.read_csv('./results/static_allocation_nd/all_regions_corr.csv', index_col=0)
mae_eu = pd.read_csv('./results/static_allocation/all_regions_mae.csv', index_col=0)

print(f'Euclidean corr: {corr_eu.shape}')
print(f'ND corr: {corr_nd.shape}')

# === Average corr ranking per method ===
methods_all = {}

# Euclidean methods (skip ITL2_average - NaN corr)
for m in corr_eu.index:
    vals = corr_eu.loc[m, STUDY_REGIONS].astype(float).dropna()
    if len(vals) > 0:
        methods_all[m] = {'mean_corr': vals.mean(), 'median_corr': vals.median(),
                          'min_corr': vals.min(), 'max_corr': vals.max(), 'source': 'euclidean'}

# ND methods
for m in corr_nd.index:
    if m not in methods_all:  # skip baselines already counted
        vals = corr_nd.loc[m, STUDY_REGIONS].astype(float).dropna()
        if len(vals) > 0:
            methods_all[m] = {'mean_corr': vals.mean(), 'median_corr': vals.median(),
                              'min_corr': vals.min(), 'max_corr': vals.max(), 'source': 'network_distance'}

corr_summary = pd.DataFrame(methods_all).T.sort_values('mean_corr', ascending=False)
corr_summary.index.name = 'method'
corr_summary.to_csv(OUTPUT_DIR / 'corr_analysis_summary.csv')

print('\n=== Average Correlation Ranking per Method ===')
display(corr_summary.round(4))
print(f'Saved: {OUTPUT_DIR / "corr_analysis_summary.csv"}')

In [ ]:
# Cell 8: Impact of GPM on Correlation + Impact of ND on Correlation

# === GPM corr delta (per region) ===
gpm_corr_rows = []
for region in STUDY_REGIONS:
    row = {'region': region}
    # EU: voronoi → voronoi_gpm
    try:
        row['voronoi_to_gpm_eu'] = float(corr_eu.loc['voronoi_gpm', region]) - float(corr_eu.loc['voronoi', region])
    except:
        row['voronoi_to_gpm_eu'] = np.nan
    # EU: civd → civd_gpm
    try:
        row['civd_to_gpm_eu'] = float(corr_eu.loc['civd_gpm', region]) - float(corr_eu.loc['civd', region])
    except:
        row['civd_to_gpm_eu'] = np.nan
    # ND: voronoi_ND → voronoi_gpm_ND
    try:
        row['voronoi_to_gpm_nd'] = float(corr_nd.loc['voronoi_gpm_ND', region]) - float(corr_nd.loc['voronoi_ND', region])
    except:
        row['voronoi_to_gpm_nd'] = np.nan
    # ND: civd_ND → civd_gpm_ND
    try:
        row['civd_to_gpm_nd'] = float(corr_nd.loc['civd_gpm_ND', region]) - float(corr_nd.loc['civd_ND', region])
    except:
        row['civd_to_gpm_nd'] = np.nan

    deltas = [v for k, v in row.items() if k != 'region' and not np.isnan(v)]
    row['avg_corr_delta'] = np.mean(deltas) if deltas else np.nan
    gpm_corr_rows.append(row)

gpm_corr_df = pd.DataFrame(gpm_corr_rows).set_index('region')
gpm_corr_df.to_csv(OUTPUT_DIR / 'gpm_corr_delta_table.csv')

print('=== Per-Region Impact of GPM on Correlation ===')
print('Positive = GPM increased corr, negative = GPM decreased corr')
display(gpm_corr_df.round(4))

# GPM corr degradation vs RMSE degradation cross-analysis
print('\n--- GPM: corr change in RMSE-degraded regions ---')
rmse_degraded = [r for r, d in gpm_delta.items() if d > 0]
for r in rmse_degraded:
    rmse_d = gpm_delta[r]
    corr_d = gpm_corr_df.loc[r, 'avg_corr_delta']
    direction = 'also degraded' if corr_d < 0 else 'improved instead'
    print(f'  {r}: RMSE delta={rmse_d:+.3f}, corr delta={corr_d:+.4f} → corr {direction}')

# === ND corr delta (per region) ===
nd_corr_rows = []
for region in STUDY_REGIONS:
    row = {'region': region}
    try:
        row['voronoi_nd_delta'] = float(corr_nd.loc['voronoi_ND', region]) - float(corr_eu.loc['voronoi', region])
    except:
        row['voronoi_nd_delta'] = np.nan
    try:
        row['civd_nd_delta'] = float(corr_nd.loc['civd_ND', region]) - float(corr_eu.loc['civd', region])
    except:
        row['civd_nd_delta'] = np.nan
    try:
        row['voronoi_gpm_nd_delta'] = float(corr_nd.loc['voronoi_gpm_ND', region]) - float(corr_eu.loc['voronoi_gpm', region])
    except:
        row['voronoi_gpm_nd_delta'] = np.nan
    try:
        row['civd_gpm_nd_delta'] = float(corr_nd.loc['civd_gpm_ND', region]) - float(corr_eu.loc['civd_gpm', region])
    except:
        row['civd_gpm_nd_delta'] = np.nan

    deltas = [v for k, v in row.items() if k != 'region' and not np.isnan(v)]
    row['avg_corr_delta'] = np.mean(deltas) if deltas else np.nan
    nd_corr_rows.append(row)

nd_corr_df = pd.DataFrame(nd_corr_rows).set_index('region')
nd_corr_df.to_csv(OUTPUT_DIR / 'nd_corr_delta_table.csv')

print('\n=== Per-Region Impact of ND on Correlation ===')
print('Positive = ND increased corr, negative = ND decreased corr')
display(nd_corr_df.round(4))

# ND corr degradation vs RMSE degradation cross-analysis
print('\n--- ND: corr change in RMSE-degraded regions ---')
nd_rmse_degraded = [r for r, d in nd_delta.items() if d > 0]
for r in nd_rmse_degraded:
    rmse_d = nd_delta[r]
    corr_d = nd_corr_df.loc[r, 'avg_corr_delta']
    direction = 'also degraded' if corr_d < 0 else 'improved instead'
    print(f'  {r}: RMSE delta={rmse_d:+.3f}, corr delta={corr_d:+.4f} → corr {direction}')

In [ ]:
# Cell 9: CIVD vs Voronoi corr comparison + corr vs RMSE scatter relationship

import matplotlib.pyplot as plt

# === CIVD vs Voronoi corr comparison ===
print('=== CIVD vs Voronoi Correlation Comparison ===\n')
civd_wins_corr = 0
for region in STUDY_REGIONS:
    v_corr = float(corr_eu.loc['voronoi', region]) if not pd.isna(corr_eu.loc['voronoi', region]) else np.nan
    c_corr = float(corr_eu.loc['civd', region]) if not pd.isna(corr_eu.loc['civd', region]) else np.nan
    if not np.isnan(v_corr) and not np.isnan(c_corr):
        winner = 'civd' if c_corr > v_corr else 'voronoi'
        if winner == 'civd':
            civd_wins_corr += 1
        print(f'  {region:8s}: voronoi={v_corr:+.4f}, civd={c_corr:+.4f} → {winner} (delta={c_corr - v_corr:+.4f})')
print(f'\nCIVD wins: {civd_wins_corr}/16')

# === corr vs RMSE scatter ===
rmse_eu = pd.read_csv('./results/static_allocation/all_regions_rmse.csv', index_col=0)

# Collect (method, region) corr vs RMSE pairs
scatter_data = []
for m in corr_eu.index:
    for r in STUDY_REGIONS:
        c_val = corr_eu.loc[m, r]
        r_val = rmse_eu.loc[m, r] if m in rmse_eu.index else np.nan
        if not pd.isna(c_val) and not pd.isna(r_val):
            scatter_data.append({'method': m, 'region': r, 'corr': float(c_val), 'rmse': float(r_val)})

scatter_df = pd.DataFrame(scatter_data)

# Overall corr vs RMSE correlation
overall_corr, overall_p = stats.pearsonr(scatter_df['corr'], scatter_df['rmse'])
print(f'\n=== Overall corr vs RMSE Relationship ===')
print(f'Pearson r = {overall_corr:.4f}, p = {overall_p:.6f}')
print(f'N = {len(scatter_df)} (method, region) points')
print(f'{"Significant negative correlation" if overall_corr < -0.3 and overall_p < 0.01 else "Weak relationship"}')

# Find anomalies with low corr but low RMSE (corr < 0.1, RMSE < median)
rmse_median = scatter_df['rmse'].median()
anomalies = scatter_df[(scatter_df['corr'] < 0.1) & (scatter_df['rmse'] < rmse_median)]
if len(anomalies) > 0:
    print(f'\nAnomalies with low corr (<0.1) but low RMSE (<{rmse_median:.1f}):')
    display(anomalies.sort_values('rmse').head(10))

# # Scatter plot
# fig, ax = plt.subplots(figsize=(8, 6))
# ax.scatter(scatter_df['corr'], scatter_df['rmse'], alpha=0.3, s=15)
# ax.set_xlabel('Pearson Correlation')
# ax.set_ylabel('RMSE (MVA)')
# ax.set_title(f'Correlation vs RMSE (r={overall_corr:.3f}, p={overall_p:.4f})')
# ax.axvline(x=0, color='gray', linestyle='--', alpha=0.3)
# plt.tight_layout()
# plt.savefig(OUTPUT_DIR / 'corr_vs_rmse_scatter.png', dpi=150, bbox_inches='tight')
# plt.show()
#
# print(f'\nChart saved: {OUTPUT_DIR / "corr_vs_rmse_scatter.png"}')